In [ ]:
from collections import deque

# --- Bước 1: Định nghĩa Miền Kế hoạch (Planning Domain) ---
#
# Mô hình hóa các hành động y tế (Action Schemas)
# Đây là "Tri thức y khoa" của hệ thống
CAC_HANH_DONG_Y_TE = {
    "KhamLamSang": {
        "mota": "Bác sĩ khám tổng quát, ghi nhận triệu chứng",
        "DIEUKIEN": {"BệnhNhânNhậpViện"},
        "HIEUUNG_THEM": {"ĐãKhamLamSang"},
        "HIEUUNG_XOA": {}
    },
    "ChiDinhXetNghiemMau": {
        "mota": "Chỉ định lấy máu để xét nghiệm công thức máu, CRP...",
        "DIEUKIEN": {"ĐãKhamLamSang"},
        "HIEUUNG_THEM": {"CóKếtQuảMáu"},
        "HIEUUNG_XOA": {}
    },
    "ChiDinhChupXQuang": {
        "mota": "Chỉ định chụp X-quang phổi",
        "DIEUKIEN": {"ĐãKhamLamSang", "KhóThở"}, # Chỉ chụp khi có triệu chứng hô hấp
        "HIEUUNG_THEM": {"CóKếtQuảXQuang"},
        "HIEUUNG_XOA": {}
    },
    "ChanDoanBenh": {
        "mota": "Tổng hợp kết quả và đưa ra chẩn đoán cuối cùng",
        # Giả sử cần cả 2 kết quả mới chẩn đoán được
        "DIEUKIEN": {"CóKếtQuảMáu", "CóKếtQuảXQuang"}, 
        "HIEUUNG_THEM": {"CóChẩnĐoán:ViêmPhổi"}, # Giả định chẩn đoán là Viêm Phổi
        "HIEUUNG_XOA": {}
    },
    "KeDonKhangSinh": {
        "mota": "Kê đơn kháng sinh điều trị Viêm Phổi",
        "DIEUKIEN": {"CóChẩnĐoán:ViêmPhổi"},
        "HIEUUNG_THEM": {"ĐãDùngKhángSinh"},
        "HIEUUNG_XOA": {}
    },
    "DieuTriTichCuc": {
        "mota": "Theo dõi và điều trị (mô phỏng quá trình bệnh nhân hồi phục)",
        "DIEUKIEN": {"ĐãDùngKhángSinh", "SốtCao", "KhóThở"},
        "HIEUUNG_THEM": {"HếtSốt", "HếtKhóThở"},
        "HIEUUNG_XOA": {"SốtCao", "KhóThở"} # Các triệu chứng biến mất
    },
    "DanhGiaLai": {
        "mota": "Đánh giá lại tình trạng sau khi hết triệu chứng",
        "DIEUKIEN": {"HếtSốt", "HếtKhóThở"},
        "HIEUUNG_THEM": {"BệnhĐãKhỏi"},
        "HIEUUNG_XOA": {}
    },
    "LamThuTucXuatVien": {
        "mota": "Làm thủ tục cho bệnh nhân xuất viện",
        "DIEUKIEN": {"BệnhĐãKhỏi"},
        "HIEUUNG_THEM": {"BệnhNhânXuấtViện"},
        "HIEUUNG_XOA": {"BệnhNhânNhậpViện"}
    }
}

# --- Bước 2: Định nghĩa Mục tiêu (Goal State) ---
MUC_TIEU = {"BệnhNhânXuấtViện", "BệnhĐãKhỏi"}

# --- Bước 3: Triển khai Thuật toán Lập kế hoạch (Forward Search) ---
# (Sử dụng lại hàm 'tim_ke_hoach' từ ví dụ trước, vì thuật toán là chung)

def tim_ke_hoach(trang_thai_dau, muc_tieu, hanh_dong):
    """
    Thực hiện tìm kiếm tiến (Forward State-Space Search) bằng BFS
    để tìm một chuỗi hành động đi từ 'trang_thai_dau' đến 'muc_tieu'.
    """
    
    print("\n Bắt đầu lập kế hoạch phác đồ điều trị...")
    
    # Hàng đợi (queue) cho BFS, mỗi phần tử là (trạng thái, kế hoạch)
    hang_doi = deque([(trang_thai_dau, [])])
    
    # 'visited' để tránh lặp lại các trạng thái đã xét
    da_ghe_tham = {frozenset(trang_thai_dau)}

    while hang_doi:
        trang_thai_hien_tai, ke_hoach_hien_tai = hang_doi.popleft()

        # --- KIỂM TRA MỤC TIÊU ---
        if muc_tieu.issubset(trang_thai_hien_tai):
            print("Lập kế hoạch thành công!")
            return ke_hoach_hien_tai 

        # --- TÌM HÀNH ĐỘNG KHẢ THI ---
        for ten_hd, dinh_nghia_hd in hanh_dong.items():
            # Kiểm tra xem điều kiện tiên quyết (PRECOND) có được đáp ứng không
            if dinh_nghia_hd["DIEUKIEN"].issubset(trang_thai_hien_tai):
                
                trang_thai_moi = trang_thai_hien_tai.copy()
                trang_thai_moi.difference_update(dinh_nghia_hd["HIEUUNG_XOA"])
                trang_thai_moi.update(dinh_nghia_hd["HIEUUNG_THEM"])

                if frozenset(trang_thai_moi) not in da_ghe_tham:
                    ke_hoach_moi = ke_hoach_hien_tai + [ten_hd]
                    hang_doi.append((trang_thai_moi, ke_hoach_moi))
                    da_ghe_tham.add(frozenset(trang_thai_moi))

    print("Không tìm thấy kế hoạch nào phù hợp.")
    return None

# --- Bước 4: Tương tác với Người dùng (Bác sĩ/Điều dưỡng) ---

print("HỆ THỐNG HỖ TRỢ QUYẾT ĐỊNH LÂM SÀNG (Mô phỏng)")
print("----------------------------------------------------------")
print("MỤC TIÊU (Goal) của phác đồ là:")
print(f"  {MUC_TIEU}")

# Liệt kê các triệu chứng/trạng thái có thể có
cac_su_that_y_te = {
    "BệnhNhânNhậpViện", "SốtCao", "KhóThở", "HoKhan", 
    "ĐãKhamLamSang", "CóKếtQuảMáu", "CóKếtQuảXQuang",
    "CóChẩnĐoán:ViêmPhổi", "ĐãDùngKhángSinh", "HếtSốt", 
    "HếtKhóThở", "BệnhĐãKhỏi", "BệnhNhânXuấtViện"
}

print(f"\nCác 'sự thật' / 'triệu chứng' y tế có thể có:")
print(f"  {', '.join(sorted(cac_su_that_y_te))}")
print("\n----------------------------------------------------------")
print("TRẠNG THÁI BAN ĐẦU (Init): Nhập tình trạng bệnh nhân")
print("Ví dụ: BệnhNhânNhậpViện, SốtCao, KhóThở")
input_str = input(">> Nhập các triệu chứng/trạng thái ban đầu, cách nhau bằng dấu phẩy: ")

# Xử lý input của người dùng thành một 'set' trạng thái
trang_thai_ban_dau = set(item.strip() for item in input_str.split(',') if item.strip())

print(f"\nTrạng thái bắt đầu của bệnh nhân: {trang_thai_ban_dau}")

# Chạy bộ lập kế hoạch
ke_hoach = tim_ke_hoach(trang_thai_ban_dau, MUC_TIEU, CAC_HANH_DONG_Y_TE)

# In kết quả
if ke_hoach:
    print("\n PHÁC ĐỒ ĐIỀU TRỊ (PLAN) ĐƯỢC ĐỀ XUẤT:")
    for i, buoc in enumerate(ke_hoach, 1):
        print(f"  Bước {i}: {buoc} ({CAC_HANH_DONG_Y_TE[buoc]['mota']})")
else:
    print("Không thể đạt được mục tiêu từ trạng thái ban đầu.")
    print("Vui lòng kiểm tra lại (ví dụ: bệnh nhân đã 'BệnhNhânNhậpViện' chưa?)")

HỆ THỐNG HỖ TRỢ QUYẾT ĐỊNH LÂM SÀNG (Mô phỏng)
----------------------------------------------------------
MỤC TIÊU (Goal) của phác đồ là:
  {'BệnhNhânXuấtViện', 'BệnhĐãKhỏi'}

Các 'sự thật' / 'triệu chứng' y tế có thể có:
  BệnhNhânNhậpViện, BệnhNhânXuấtViện, BệnhĐãKhỏi, CóChẩnĐoán:ViêmPhổi, CóKếtQuảMáu, CóKếtQuảXQuang, HoKhan, HếtKhóThở, HếtSốt, KhóThở, SốtCao, ĐãDùngKhángSinh, ĐãKhamLamSang

----------------------------------------------------------
TRẠNG THÁI BAN ĐẦU (Init): Nhập tình trạng bệnh nhân
Ví dụ: BệnhNhânNhậpViện, SốtCao, KhóThở
